# Titanic — ensemble pipeline (strong public LB)

**Ideas:** train-only statistics (no leakage), rich features (deck, ticket frequency, fare/person), **out-of-fold stacking** with multiple base learners, final refit on full training data.

Paths assume Kaggle (`/kaggle/input/titanic/`). Locally, set `DATA_PATH` to your CSV folder.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

import os
if os.path.exists("/kaggle/input/titanic"):
    DATA_PATH = "/kaggle/input/titanic"
elif os.path.exists("/kaggle/input/competitions/titanic"):
    DATA_PATH = "/kaggle/input/competitions/titanic"
elif os.path.exists("/kaggle/input/titanic-competition"):
    DATA_PATH = "/kaggle/input/titanic-competition"
else:
    DATA_PATH = "."

train = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
test = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

TARGET = "Survived"
ID_COL = "PassengerId"
n_train = len(train)

## Feature engineering

- Impute `Age` with median by `Sex` and `Pclass` (grouped fill).
- `Fare` per person, `Deck` from `Cabin`, ticket frequency (counts from **train** only).
- `FareBin` edges from **train** quantiles only.

In [ ]:
def extract_title(name):
    return name.split(",")[1].split(".")[0].strip()

def add_features(train_df, test_df):
    train = train_df.copy()
    test = test_df.copy()
    full = pd.concat(
        [train.drop(columns=[TARGET], errors="ignore"), test],
        axis=0,
        sort=False,
        ignore_index=True,
    )

    full["Title"] = full["Name"].apply(extract_title)
    rare = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}
    full["Title"] = full["Title"].replace(list(rare), "Rare")
    full["Title"] = full["Title"].replace({"Mlle": "Miss", "Mme": "Mrs", "Ms": "Miss"})

    full["FamilySize"] = full["SibSp"] + full["Parch"] + 1
    full["IsAlone"] = (full["FamilySize"] == 1).astype(int)
    full["FarePerPerson"] = full["Fare"] / full["FamilySize"]

    full["Deck"] = full["Cabin"].apply(lambda x: x[0] if pd.notna(x) else "U")
    full["HasCabin"] = full["Cabin"].notna().astype(int)

    tick_train = full.iloc[:n_train]["Ticket"].value_counts()
    full["TicketFreq"] = full["Ticket"].map(tick_train).fillna(1)

    full["Fare"] = full.groupby("Pclass")["Fare"].transform(lambda s: s.fillna(s.median()))
    full["Fare"] = full["Fare"].fillna(full["Fare"].median())
    full["FareLog"] = np.log1p(full["Fare"])
    full["FarePerPerson"] = full["FarePerPerson"].fillna(
        full.groupby("Pclass")["FarePerPerson"].transform("median")
    )
    full["FarePerPerson"] = full["FarePerPerson"].fillna(full["FarePerPerson"].median())

    age_guess = full.groupby(["Sex", "Pclass"])["Age"].transform("median")
    full["Age"] = full["Age"].fillna(age_guess)
    full["Age"] = full["Age"].fillna(full["Age"].median())

    full["IsChild"] = (full["Age"] < 16).astype(int)
    full["Sex"] = full["Sex"].map({"male": 0, "female": 1})

    emb = pd.get_dummies(full["Embarked"].fillna(full["Embarked"].mode()[0]), prefix="Emb")
    full = pd.concat([full, emb], axis=1)

    tr_fare = full.iloc[:n_train]["FareLog"]
    try:
        _, fare_bins = pd.qcut(tr_fare, q=6, retbins=True, duplicates="drop")
    except Exception:
        _, fare_bins = pd.qcut(tr_fare, q=4, retbins=True, duplicates="drop")
    full["FareBin"] = pd.cut(full["FareLog"], bins=fare_bins, labels=False, include_lowest=True)
    full["FareBin"] = full["FareBin"].astype(float).fillna(0).astype(int)

    full["AgeBin"] = pd.cut(full["Age"], bins=[0, 6, 12, 18, 35, 60, 100], labels=False)

    tr_slice = full.iloc[:n_train]
    title_cats = sorted(set(tr_slice["Title"].astype(str).unique()) | {"Rare"})
    deck_cats = sorted(tr_slice["Deck"].astype(str).unique())
    t_str = full["Title"].astype(str)
    d_str = full["Deck"].astype(str)
    t_str = t_str.where(t_str.isin(title_cats), other="Rare")
    d_str = d_str.where(d_str.isin(deck_cats), other="U")
    full["Title"] = pd.Categorical(t_str, categories=title_cats).codes
    full["Deck"] = pd.Categorical(d_str, categories=deck_cats).codes

    full["Pclass_Sex"] = full["Pclass"] * full["Sex"]
    full["Age_Pclass"] = full["Age"] * full["Pclass"]

    feature_cols = [
        "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "FareLog", "FarePerPerson",
        "FamilySize", "IsAlone", "HasCabin", "TicketFreq", "Title", "Deck",
        "FareBin", "AgeBin", "IsChild", "Pclass_Sex", "Age_Pclass",
    ] + [c for c in full.columns if c.startswith("Emb_")]

    X_tr = full.iloc[:n_train][feature_cols].copy()
    X_te = full.iloc[n_train:][feature_cols].copy()

    for c in X_tr.columns:
        X_tr[c] = pd.to_numeric(X_tr[c], errors="coerce").fillna(0)
        X_te[c] = pd.to_numeric(X_te[c], errors="coerce").fillna(0)

    return X_tr, X_te, feature_cols

X, X_test, FEATURES = add_features(train, test)
y = train[TARGET].values

print("Shape:", X.shape, X_test.shape)
print("Features:", len(FEATURES))

## Stacked ensemble (OOF + meta-learner)

Base models: **XGBoost**, **LightGBM** (if installed), **RandomForest**, **GradientBoosting**. Meta: **logistic regression** on OOF probabilities.

Final: refit bases on **all** training rows, then meta on test.

In [ ]:
def get_models(seed):
    models = {
        "xgb": XGBClassifier(
            n_estimators=800,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=2,
            subsample=0.85,
            colsample_bytree=0.85,
            gamma=0.05,
            reg_alpha=0.05,
            reg_lambda=1.0,
            eval_metric="logloss",
            random_state=seed,
            n_jobs=-1,
        ),
        "rf": RandomForestClassifier(
            n_estimators=500,
            max_depth=8,
            min_samples_leaf=2,
            min_samples_split=4,
            max_features="sqrt",
            random_state=seed,
            n_jobs=-1,
        ),
        "gb": GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            min_samples_leaf=2,
            random_state=seed,
        ),
    }
    if LGBMClassifier is not None:
        models["lgb"] = LGBMClassifier(
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=seed,
            n_jobs=-1,
            verbose=-1,
        )
    return models

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_list = []
test_stack = []

model_names = list(get_models(0).keys())

for name in model_names:
    oof_col = np.zeros(n_train)
    test_fold = np.zeros((len(test), N_SPLITS))
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        clf = get_models(42 + fold)[name]
        clf.fit(X.iloc[tr_idx], y[tr_idx])
        oof_col[val_idx] = clf.predict_proba(X.iloc[val_idx])[:, 1]
        test_fold[:, fold] = clf.predict_proba(X_test)[:, 1]
    oof_list.append(oof_col.reshape(-1, 1))
    test_stack.append(test_fold.mean(axis=1).reshape(-1, 1))

OOF = np.hstack(oof_list)
TEST_BASE = np.hstack(test_stack)
print("OOF shape:", OOF.shape, "Test base:", TEST_BASE.shape)

scaler = StandardScaler()
OOF_s = scaler.fit_transform(OOF)
meta = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
meta.fit(OOF_s, y)
oof_pred = meta.predict(OOF_s)
print("Stack OOF accuracy:", accuracy_score(y, oof_pred))

final_bases = []
for name in model_names:
    m = get_models(99)[name]
    m.fit(X, y)
    final_bases.append(m.predict_proba(X_test)[:, 1].reshape(-1, 1))

TEST_FULL = np.hstack(final_bases)
TEST_FULL_s = scaler.transform(TEST_FULL)
final_survived = (meta.predict_proba(TEST_FULL_s)[:, 1] >= 0.5).astype(int)

sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: final_survived})
sub.to_csv("submission.csv", index=False)
print("Saved submission.csv")

## Notes

- **0.85 on the public leaderboard** is not something any notebook can honestly promise: the test set is small and scores jump with variance. **0.80–0.83** is a realistic “strong” band; this matches typical top-kernel stack + features.
- Install **LightGBM** if missing: `pip install lightgbm`.
- Optional: average multiple **random seeds** for base models and stack again for a small boost.